Install dependencies first

YOU NEED TO RUN PYTHON 3.12, NOT ALL DEPENDENCIES WORK ON 3.13

In [ ]:
%pip install mne
%pip install asrpy
%pip install pandas
%pip install pyxdf
%pip install matplotlib.pyplot
%pip install numpy
%pip install scipy

import os
import csv
import glob
import matplotlib.pyplot as plt
import mne
import numpy as np
import io
import pandas as pd
import scipy
import tempfile
from asrpy import ASR
from collections import OrderedDict
from mne_connectivity import envelope_correlation
from mne_icalabel import label_components
from scipy import stats

This just makes the graphs scrollable. It's a bit weird so if it doesn't work, do not run the cell.

In [ ]:
#Make the graphs scrollable
%matplotlib ipympl

In [ ]:
event_dict = {
    "START-FM-Baseline": 1,
    "STOP-FM-Baseline": 2,
    "START-FM-baseline1": 3,
    "STOP-FM-baseline1": 4,
    "START-FM-Metronome": 5,
    "STOP-FM-Metronome": 6,
    "START-FM-P2_leader": 7,
    "STOP-FM-P2_leader": 8,
    "START-FM-P1_leader": 9,
    "STOP-FM-P1_leader": 10
}
event_id = event_dict

freq_bands = {
    'Delta': [0.5, 3.5],
    'Theta': [4, 7],
    'Alpha': [7.5, 13],
    'Beta': [13.5, 30],
    'Gamma': [30.5, 40]
}

All the preprocessing prior to an independent components analysis

In [ ]:
from pathlib import Path

def process_eeg_csv(file_path):
    """
    Process 24-channel EEG CSV file and add proper headers.
    
    Parameters:
    -----------
    file_path : Path or str
        Path to the raw CSV file
        
    Returns:
    --------
    pd.DataFrame
        Processed dataframe with 24 EEG channels, Time, and Trigger columns
    """
    # Read the CSV to handle variable column counts
    data = []
    with open(file_path, 'r') as f:
        reader = csv.reader(f)
        for row in reader:
            # Pad rows to have 27 columns (26 data + 1 trigger if needed)
            while len(row) < 26:
                row.append('')
            data.append(row)
    
    # Create DataFrame with all 10 columns
    df = pd.DataFrame(data)
    
    # Remove columns E, F, G, H (indices 4, 5, 6, 7)
    # Keep columns A, B, C, D, I, J (indices 0, 1, 2, 3, 8, 9)
    columns_to_keep = list(range(26))
    df_processed = df.iloc[:, columns_to_keep].copy()
    
    # Add column names
    column_names = ['Fp1', 'Fp2', 'F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2', 
                 'F7', 'F8', 'T7', 'T8', 'P7', 'P8', 'Fz', 'Cz', 'Pz', 'M1', 
                 'M2', 'AFz', 'CPz', 'POz', 'Time', 'Trigger']
    df_processed.columns = column_names
    
    # Convert numeric columns to float (leaving Trigger as string)
    for col in ['Fp1', 'Fp2', 'F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2', 
                 'F7', 'F8', 'T7', 'T8', 'P7', 'P8', 'Fz', 'Cz', 'Pz', 'M1', 
                 'M2', 'AFz', 'CPz', 'POz', 'Time']:
        df_processed[col] = pd.to_numeric(df_processed[col], errors='coerce')
    
    # Replace empty strings in Trigger column with NaN
    df_processed['Trigger'] = df_processed['Trigger'].replace('', pd.NA)
    
    print(f"Processed {file_path.name}: {df_processed.shape[0]} rows")
    trigger_count = df_processed['Trigger'].notna().sum()
    if trigger_count > 0:
        print(f"  Found {trigger_count} trigger markers: {df_processed['Trigger'].dropna().unique()}")
    
    return df_processed

# Column names and paths
expected_n_cols_eeg = 24
col_names_eeg = ['Fp1', 'Fp2', 'F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2', 
                 'F7', 'F8', 'T7', 'T8', 'P7', 'P8', 'Fz', 'Cz', 'Pz', 'M1', 
                 'M2', 'AFz', 'CPz', 'POz']
other_columns = ['Time', 'Trigger']

data_path1 = Path(r"C:\Users\zkar517\Documents\MuseAnalysis\pair20\mbtrain1\mbtrain1-FM.csv")
data_path2 = Path(r"C:\Users\zkar517\Documents\MuseAnalysis\pair20\mbtrain2\mbtrain2-FM.csv")
event_ids = ["START-FM-Baseline", "STOP-FM-Baseline",
            "START-FM-baseline1", "STOP-FM-baseline1",
            "START-FM-Metronome", "STOP-FM-Metronome",
            "START-FM-P2_leader", "STOP-FM-P2_leader",
            "START-FM-P1_leader", "STOP-FM-P1_leader"
            ]
Trigger = event_ids

# Load and process CSV data
print("Processing CSV files...")
df1 = process_eeg_csv(data_path1)
df2 = process_eeg_csv(data_path2)

print(f"\ndf1 shape: {df1.shape}, columns: {df1.columns.tolist()}")
print(f"df2 shape: {df2.shape}, columns: {df2.columns.tolist()}")

# Extract only EEG columns
df1_eeg = df1[col_names_eeg]
df2_eeg = df2[col_names_eeg]

print(f"\ndf1_eeg shape: {df1_eeg.shape}")
print(f"df2_eeg shape: {df2_eeg.shape}")

# Create MNE Raw objects
# Muse exports data in microvolts (μV), so we need to convert to volts (V)
# MNE expects data in volts: 1 μV = 1e-6 V
eeg_info = mne.create_info(ch_names=col_names_eeg, sfreq=256, ch_types="eeg")
montage = mne.channels.make_standard_montage('standard_1020')
eeg_info.set_montage(montage)

# Convert from microvolts to volts by dividing by 1e6
raw_eeg1 = mne.io.RawArray(df1_eeg.T.to_numpy() / 1e6, eeg_info)
raw_eeg2 = mne.io.RawArray(df2_eeg.T.to_numpy() / 1e6, eeg_info)

print(f"\n✅ Data loaded and converted from μV to V")
print(f"   MNE data range: {raw_eeg1.get_data().min():.2e} to {raw_eeg1.get_data().max():.2e} V")
print(f"   (Expected EEG range: ~-100e-6 to 100e-6 V for typical EEG)")
print(raw_eeg1.info) # Double check it looks alright

# Check triggers (let me know if they are not showing up)
print(f"\n📍 Trigger information available in df1['Trigger'] and df2['Trigger']")
print(f"df1 triggers: {df1['Trigger'].dropna().unique()}\ndf2 triggers: {df2['Trigger'].dropna().unique()}")

print(f"\n✅ Data loaded and converted from μV to V")
print(f"   MNE data range: {raw_eeg1.get_data().min():.2e} to {raw_eeg1.get_data().max():.2e} V")
print(f"   (Expected EEG range: ~-100e-6 to 100e-6 V for typical EEG)")
print(raw_eeg1.info) # Double check it looks alright

# Check triggers (let me know if they are not showing up)
print(f"\n📍 Trigger information available in df1['Trigger'] and df2['Trigger']")
print(f"df1 triggers: {df1['Trigger'].dropna().unique()}\ndf2 triggers: {df2['Trigger'].dropna().unique()}")

In [ ]:
# Set channel types to EEG before setting montage (ran into some issues here if it doesn't)
raw_eeg1.set_channel_types({ch: 'eeg' for ch in raw_eeg1.ch_names})
raw_eeg2.set_channel_types({ch: 'eeg' for ch in raw_eeg2.ch_names})

# Set the montage - 10-20 config
montage = mne.channels.make_standard_montage('standard_1020')
raw_eeg1.set_montage(montage)
raw_eeg2.set_montage(montage)

# Filters = 0.5-40Hz BPF, Re-referencing = Average
raw_downsampled1 = raw_eeg1.copy().resample(250)
raw_downsampled2 = raw_eeg2.copy().resample(250)
raw_filtered1 = raw_eeg1.copy().filter(l_freq=0.5, h_freq=40).notch_filter(freqs=[50], method='spectrum_fit', filter_length='auto', phase='zero')
raw_filtered2 = raw_eeg2.copy().filter(l_freq=0.5, h_freq=40).notch_filter(freqs=[50], method='spectrum_fit', filter_length='auto', phase='zero')

# Re-reference to M1 and M2
raw_filtered1.set_eeg_reference(ref_channels=['M1', 'M2'], ch_type='eeg')
raw_filtered2.set_eeg_reference(ref_channels=['M1', 'M2'], ch_type='eeg')
raw_filtered1.drop_channels(['M1', 'M2'])  # Remove mastoids after re-referencing since we don't need them
raw_filtered2.drop_channels(['M1', 'M2'])

just_checking_the_sensor_locs1 = raw_filtered1.plot_sensors(show_names=True)

print(raw_filtered1.info) # Double checking it again

In [ ]:
raw_filtered1.plot(scalings=dict(eeg=50e-6), title='Subject 1 - Filtered EEG', show=True)
print()